# Minifigures Application Deployment to AWS EC2

This notebook automates the deployment of the Minifigures Docker application to AWS EC2. Run each cell in order.

**Prerequisites:**
- Docker installed locally
- AWS CLI configured
- SSH key pair available at `C:\Users\Usuario\Desktop\Realization of AI\fresca-lorenzo-key-pair.pem`
- EC2 instance running with IP: `35.180.39.26`

## Step 1: Build Docker Image Locally

Run this command to build the Docker image with no cache (ensures fresh build with latest code)

In [1]:
import subprocess
import os

# Change to workspace directory
os.chdir("/workspaces/updated-minifigures-webshop-2026-LorenzSF")

# Build Docker image
print("Building Docker image...")
result = subprocess.run(
    [
        "docker",
        "build",
        "--no-cache",
        "-t",
        "516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest",
        ".",
    ],
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print("✓ Docker image built successfully!")
else:
    print("✗ Docker build failed!")

Building Docker image...


#0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 4.47kB 0.1s done
#1 DONE 0.1s

#2 resolve image config for docker-image://docker.io/docker/dockerfile:1
#2 ...

#3 [auth] docker/dockerfile:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 resolve image config for docker-image://docker.io/docker/dockerfile:1
#2 DONE 1.2s

#4 docker-image://docker.io/docker/dockerfile:1@sha256:4a43a54dd1fedceb30ba47e76cfcf2b47304f4161c0caeac2db1c61804ea3c91
#4 resolve docker.io/docker/dockerfile:1@sha256:4a43a54dd1fedceb30ba47e76cfcf2b47304f4161c0caeac2db1c61804ea3c91 0.0s done
#4 CACHED

#5 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#5 ...

#6 [auth] library/python:pull token for registry-1.docker.io
#6 DONE 0.0s

#7 [internal] load metadata for docker.io/library/python:3.10-slim
#7 DONE 0.9s

#5 [internal] load metadata for ghcr.io/astral-sh/uv:latest
#5 DONE 0.9s

#8 [internal] load .dockerignore
#

✓ Docker image built successfully!


#19 DONE 219.7s


## Step 2: Push Docker Image to ECR

Authentication on ECR of AWS

In [5]:
import subprocess

# Authenticate with ECR
print("Authenticating with ECR...")
auth_result = subprocess.run(
    ["aws", "ecr", "get-login-password", "--region", "eu-west-3"], capture_output=True, text=True
)

docker_login = subprocess.run(
    [
        "docker",
        "login",
        "--username",
        "AWS",
        "--password-stdin",
        "516454187396.dkr.ecr.eu-west-3.amazonaws.com",
    ],
    input=auth_result.stdout,
    text=True,
)

if docker_login.returncode == 0:
    print("✓ ECR authentication successful!")
else:
    print("✗ ECR authentication failed!")

Authenticating with ECR...
Login Succeeded
✓ ECR authentication successful!


Push the built image to your ECR repository

In [6]:
import subprocess

# Push image to ECR
print("Pushing Docker image to ECR...")
result = subprocess.run(
    ["docker", "push", "516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest"],
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print("✓ Image pushed to ECR successfully!")
else:
    print("✗ Docker push failed!")

Pushing Docker image to ECR...
The push refers to repository [516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo]
0e5d180ecc3e: Waiting
c3d226db475f: Waiting
dc2eb3bb2644: Waiting
56eb2a1e4e47: Waiting
b7c0008def3d: Waiting
ec781dee3f47: Waiting
8f1c8b5e20ba: Waiting
0833627ad603: Waiting
50bb60521db6: Waiting
a5848b3a066c: Waiting
1273c0845803: Waiting
0833627ad603: Waiting
50bb60521db6: Waiting
a5848b3a066c: Layer already exists
1273c0845803: Waiting
0e5d180ecc3e: Waiting
c3d226db475f: Layer already exists
dc2eb3bb2644: Waiting
56eb2a1e4e47: Waiting
b7c0008def3d: Waiting
ec781dee3f47: Layer already exists
8f1c8b5e20ba: Waiting
50bb60521db6: Layer already exists
1273c0845803: Waiting
0e5d180ecc3e: Waiting
0833627ad603: Pushed
dc2eb3bb2644: Pushed
0e5d180ecc3e: Pushed
1273c0845803: Pushed
56eb2a1e4e47: Pushed
8f1c8b5e20ba: Pushed
b7c0008def3d: Pushed
latest: digest: sha256:96beeaf169f2a17363f41ca4a754283b09accb37f700e8b14060c807b9c2ea53 size: 856
✓ Image pushed to ECR successfu

## Step 3: SSH into EC2 Instance

Open your terminal and run this command to connect to your EC2 instance:

```bash
ssh ec2-user@35.180.39.26
```

Then become root:
```bash
sudo su
```

Once you're on the EC2 instance (you should see `[root@ip-...]#`), proceed to the next step.

## Step 4: Complete EC2 Deployment Script

Copy and paste the following entire script into your EC2 terminal. This will execute all deployment steps at once.
It prepares a persistent runtime folder under `/root/minifigures-runtime`, downloads the public dataset, syncs model artifacts from S3, and mounts that runtime folder into the API container.

Before running it, make sure your trained model has been uploaded to your S3 bucket under `models/my_model/`.

# Authenticate with ECR

aws ecr get-login-password --region eu-west-3 | docker login --username AWS --password-stdin 516454187396.dkr.ecr.eu-west-3.amazonaws.com

```bash
#!/bin/bash
set -e

RUNTIME_ROOT=/root/minifigures-runtime
DATA_DIR=$RUNTIME_ROOT/data
MODEL_DIR=$RUNTIME_ROOT/models
MODEL_TAG=my_model
S3_BUCKET=frescalorenzo
IMAGE_URI=516454187396.dkr.ecr.eu-west-3.amazonaws.com/frescalorenzo:latest

# Step 1: Prepare runtime folders
mkdir -p $DATA_DIR $MODEL_DIR

# Step 2: Download dataset if missing
if [ ! -d $DATA_DIR/minifigures ] || [ -z "$(find $DATA_DIR/minifigures -maxdepth 1 -name '*.png' -print -quit 2>/dev/null)" ]; then
  curl -fL https://roai-data-readonly.s3.eu-central-1.amazonaws.com/minifigures.tar.gz -o $DATA_DIR/minifigures.tar.gz
  tar -xzf $DATA_DIR/minifigures.tar.gz -C $DATA_DIR
  rm -f $DATA_DIR/minifigures.tar.gz
fi
curl -fL https://roai-data-readonly.s3.eu-central-1.amazonaws.com/dataset.json -o $DATA_DIR/dataset.json

# Step 3: Sync model artifacts from S3
aws s3 cp s3://$S3_BUCKET/models/$MODEL_TAG $MODEL_DIR/$MODEL_TAG --recursive

# Step 4: Ensure Docker network exists
docker network create kulroai-net 2>/dev/null || true

# Step 5: Stop and remove old containers
docker stop api app 2>/dev/null || true
docker rm api app 2>/dev/null || true

# Step 6: Pull new image
docker pull $IMAGE_URI

# Step 7: Run API container with mounted runtime data
docker run -d --network kulroai-net --name api \
  -v $RUNTIME_ROOT:/workspaces/minifigures-app/data \
  -p 8000:8000 \
  -e PYTHONPATH=/workspaces/minifigures-app/src:$PYTHONPATH \
  $IMAGE_URI api

# Step 8: Run Streamlit app container
docker run -d --network kulroai-net --name app \
  -p 80:8500 \
  -e API_HOST=http://api \
  -e API_PORT=8000 \
  $IMAGE_URI app

echo "=== Step 9: Check container status ==="
docker ps
curl http://localhost:8000/data/get_image_tags/ | head

echo "✓ Deployment complete!"
```

## Step 5: Verify Deployment

Run these commands on your EC2 instance to check container logs and mounted artifacts:

```bash
# Check API logs
docker logs api

# Check app logs
docker logs app

# List running containers
docker ps

# Confirm dataset and model are mounted
find /root/minifigures-runtime/data/minifigures -maxdepth 1 -name '*.png' | head
find /root/minifigures-runtime/models/my_model -maxdepth 1 -type f

# Confirm the API can serve image tags
curl http://localhost:8000/data/get_image_tags/ | head
```

Then open your browser to access the application:
```
http://35.180.39.26
```

If successful, you should see the Minifigures app with Home, Product, and Market pages, and prediction requests should no longer return `404 Model 'my_model' not found`.

## Summary

**Deployment Flow:**
1. ✓ Build Docker image locally with fresh code
2. ✓ Push image to AWS ECR
3. ✓ SSH into EC2 instance
4. ✓ Prepare persistent runtime folders on EC2
5. ✓ Download dataset and dataset metadata
6. ✓ Sync model artifacts from S3
7. ✓ Pull new image from ECR
8. ✓ Run API container with mounted runtime data
9. ✓ Run Streamlit app container with API configuration
10. ✓ Access application via IP or domain

**Key Environment Variables:**
- `PYTHONPATH=/workspaces/minifigures-app/src:$PYTHONPATH` - Makes Python find the src modules
- `API_HOST=http://api` - Docker network container name
- `API_PORT=8000` - API port
- `S3_BUCKET=frescalorenzo` - Bucket containing the deployed model artifacts
- `MODEL_TAG=my_model` - Model folder expected by the API at prediction time

**Access Points:**
- IP: `http://35.180.39.26`
- Domain: `http://frescalorenzo.realization-of-ai.com`